In [31]:
import numpy as np
import matplotlib.pyplot as plt

import keras

from keras.datasets import mnist

from keras.models import Sequential

from keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout

## Get the data and pre-process it

In [32]:
(X_train, y_train), (X_test, y_test) =mnist.load_data()


X_train.shape, y_train.shape, X_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

In [33]:
def plot_input_img(i):
    plt.imshow(X_train[i], cmap= 'binary')
    plt.title(y_train[i])
    plt.axis('off')
    plt.show()

In [34]:
# Pre Process the images

# Normalizing the image [0, 1] range
if X_train.max() > 1.0:
    X_train = X_train.astype(np.float32) / 255.0 # normalize it
if X_test.max() > 1.0:
    X_test = X_test.astype(np.float32) / 255.0 # normalize it

# Reshape / expand the dimensions of images to (28, 28, 1)
if len(X_train.shape) == 3:
    X_train = np.expand_dims(X_train, -1)
if len(X_test.shape) == 3:
    X_test = np.expand_dims(X_test, -1)  # Fixed bug: was X_train instead of X_test

# Convert classes to one hot vectors
if len(y_train.shape) == 1:
    y_train = keras.utils.to_categorical(y_train)
if len(y_test.shape) == 1:
    y_test = keras.utils.to_categorical(y_test)


In [35]:
X_train.shape

(60000, 28, 28, 1)

In [36]:
y_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [37]:
model = Sequential()

model.add(Conv2D(32, (3,3), input_shape = (28, 28, 1), activation='relu'))
model.add(MaxPool2D((2,2)))

model.add(Conv2D(64, (3,3), input_shape = (28, 28, 1), activation='relu'))
model.add(MaxPool2D((2,2)))

model.add(Flatten())

model.add(Dropout(0.25))

model.add(Dense(10, activation="softmax"))

c:\Users\suyas\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.compile(optimizer='adam', loss = keras.losses.categorical_crossentropy,metrics=['accuracy'])

In [ ]:
# Callbacks
from keras.callbacks import EarlyStopping, ModelCheckpoint

# EarlyStopping
es = EarlyStopping(monitor='val_accuracy', min_delta=0.01, patience=4, verbose=1)

# Model Check Point
mc = ModelCheckpoint("./bestmodel.h5", monitor="val_accuracy", verbose=1, save_best_only=True)

cb = [es, mc]


## Model Training

In [41]:
his = model.fit(X_train, y_train, epochs=50, validation_split=0.3, callbacks=cb)


Epoch 1/50


1313/1313 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8549 - loss: 0.4636 - val_accuracy: 0.9747 - val_loss: 0.0795
Epoch 2/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9759 - loss: 0.0752 - val_accuracy: 0.9822 - val_loss: 0.0559
Epoch 3/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9839 - loss: 0.0523 - val_accuracy: 0.9836 - val_loss: 0.0540
Epoch 4/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.9870 - loss: 0.0411 - val_accuracy: 0.9857 - val_loss: 0.0475
Epoch 5/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9895 - loss: 0.0339 - val_accuracy: 0.9872 - val_loss: 0.0412
Epoch 6/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9903 - loss: 0.0298 - val_accuracy: 0.9871 - val_loss: 0.0441
Epoch 7/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9913 - loss: 0.0267 - val_accuracy: 0.9877 - val_loss: 0.0435
Epoch 8/50
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.9917 - loss: 0.0232 - val_accurac

In [42]:
model.save("bestmodel.h5")


In [44]:
model_S = keras.models.load_model("./bestmodel.h5")

In [45]:
score = model_S.evaluate(X_test, y_test)

print(f"The model accuracy is {score[1]}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9900 - loss: 0.0513
The model accuracy is 0.9912999868392944
